# Notebook 1 - Landing to Bronze

# Notebook 1 - Landing -> Bronze (Ingestao)

**Projeto:** CineData Analytics | Rocket Lab 2026
**Disciplina:** Engenharia de Dados (Databricks + PySpark + Delta)

---
## Objetivo
Criar a camada **Bronze** do Data Lakehouse: espelho fiel e imutavel dos dados brutos,
sem qualque transformacao estrutural ou de conteudo. A unica alteracao permitida e a adicao
da coluna de auditoria **`ingestion_datetime`** (timestamp exato da insercao).

---
## Entregas desta etapa
1. Criar o database `bronze`.
2. Ingerir os 5 CSVs do Volume -> tabelas Bronze (Delta, modo **Append**).
3. Ingerir a **cotacao do dolar (BRL)** via **API PTAX do Banco Central** -> `bronze.tb_cotacao_dolar`.

> Regra de ouro da Bronze: dado bruto entra EXATAMENTE como veio. Nada de limpeza aqui.


In [ ]:
# ============================================================
# 1) Configuracao (banco, formato, caminho)
# ============================================================
bronze_db      = "bronze"
table_format   = "delta"
volume_path    = "/Volumes/cinedata/default/inputs"   # ajuste ao seu Volume

spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
print(f"[OK] Database `{bronze_db}` pronto.")


---
## Mapeamento arquivo -> tabela Bronze

| Arquivo (Volume)                         | Tabela Bronze                    |
|------------------------------------------|----------------------------------|
| `movies_info_TMDB_IMDB.csv`              | `bronze.tb_movies_info`          |
| `movies_financials_IMDB_TMDB.csv`        | `bronze.tb_movies_financials`    |
| `movies_metrics_IMDB_TMDB.csv`           | `bronze.tb_movies_metrics`       |
| `credits_and_tags_IMDB_TMDB.csv`         | `bronze.tb_credits_and_tags`     |
| `movies_reviews.csv`                     | `bronze.tb_movies_reviews`       |
| API PTAX (Banco Central)                 | `bronze.tb_cotacao_dolar`        |


In [ ]:
# ============================================================
# 2) Funcao generica: CSV -> Bronze (Delta, Append)
# ============================================================
from pyspark.sql import functions as F

def ingest_to_bronze(table_name, file_name):
    src = f"{volume_path}/{file_name}"
    df = (spark.read
          .option("header", "true")
          .option("sep", ",")
          .option("inferSchema", "true")     # espelho fiel do schema inferido
          .option("multiLine", "true")
          .option("escape", '"')
          .option("mode", "PERMISSIVE")      # nao aborta em linhas com ruido
          .csv(src))

    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    (df.write
       .mode("append")                       # Append = preserva historico na Bronze
       .format(table_format)
       .option("mergeSchema", "true")
       .saveAsTable(f"{bronze_db}.{table_name}"))

    n = spark.table(f"{bronze_db}.{table_name}").count()
    print(f"[BRONZE] {bronze_db}.{table_name}: {n:,}")


---
## Instrucoes de execucao (Landing -> Bronze)
1. Substitua `volume_path` pelo caminho real do seu Volume (ex.: `/Volumes/nome/default/inputs`).
2. Execute as celulas em ordem.
3. Na Bronze **nao aplicamos negocio**: os dados ficam iguais a origem + `ingestion_datetime`.


In [ ]:
# ============================================================
# 3) Ingestao efetiva dos 5 arquivos
# ============================================================
ingest_to_bronze("tb_movies_info",       "movies_info_TMDB_IMDB.csv")
ingest_to_bronze("tb_movies_financials", "movies_financials_IMDB_TMDB.csv")
ingest_to_bronze("tb_movies_metrics",    "movies_metrics_IMDB_TMDB.csv")
ingest_to_bronze("tb_credits_and_tags",  "credits_and_tags_IMDB_TMDB.csv")
ingest_to_bronze("tb_movies_reviews",    "movies_reviews.csv")


---
## Cotacao do dolar via API PTAX (Banco Central do Brasil)
A diretoria financeira quer acompanhar orcamento e receita dos filmes tambem em **Reais (BRL)**.

**Endpoint (formato de data exigido: MM-DD-AAAA):**
```
https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/
CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)
  ?@dataInicial='MM-DD-AAAA'&@dataFinalCotacao='MM-DD-AAAA'
  &$select=dataHoraCotacao,cotacaoCompra&$format=json
```
> A API **nao retorna cotacao nos finais de semana e feriados**, por isso consultamos **os ultimos 7 dias**
> a partir da data de execucao, com **widgets** para as datas.


In [ ]:
# ============================================================
# 4) Widgets com as datas da cotacao (formato MM-DD-AAAA)
# ============================================================
from datetime import date, timedelta

hoje = date.today()
sete_dias_atras = hoje - timedelta(days=7)

# valores padrao (formato da PTAX: MM-DD-AAAA)
ini_padrao = sete_dias_atras.strftime("%m-%d-%Y")
fim_padrao = hoje.strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", ini_padrao, "data_inicio (MM-DD-AAAA)")
dbutils.widgets.text("data_fim",    fim_padrao, "data_fim    (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim    = dbutils.widgets.get("data_fim")
print(f"Janela PTAX: {data_inicio} -> {data_fim}")


In [ ]:
# ============================================================
# 5) Chamada a API PTAX e gravacao na Bronze
# ============================================================
import requests
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType)

url = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
       f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
       "&$select=dataHoraCotacao,cotacaoCompra&$format=json")

resp = requests.get(url, timeout=30)
resp.raise_for_status()
rows = resp.json().get("value", [])
print(f"Cotacoes retornadas: {len(rows)}")

if rows:
    schema = StructType([
        StructField("dataHoraCotacao", StringType(), True),
        StructField("cotacaoCompra",   DoubleType(), True),
    ])
    df_cot = (spark.createDataFrame(rows, schema=schema)
        .withColumn("data_cotacao", F.to_date("dataHoraCotacao"))
        .withColumn("cotacao_compra_brl", F.col("cotacaoCompra"))
        .withColumn("ingestion_datetime", F.current_timestamp())
        .select("data_cotacao", "cotacao_compra_brl", "ingestion_datetime"))

    (df_cot.write
       .mode("append").format(table_format)
       .saveAsTable(f"{bronze_db}.tb_cotacao_dolar"))
    n = spark.table(f"{bronze_db}.tb_cotacao_dolar").count()
    print(f"[BRONZE] {bronze_db}.tb_cotacao_dolar: {n:,}")


---
## Validacao final da camada Bronze


In [ ]:
# ============================================================
# 6) Conferencia das tabelas Bronze
# ============================================================
tabelas = [
    "tb_movies_info", "tb_movies_financials", "tb_movies_metrics",
    "tb_credits_and_tags", "tb_movies_reviews", "tb_cotacao_dolar",
]
for t in tabelas:
    try:
        n = spark.table(f"{bronze_db}.{t}").count()
        print(f"  {bronze_db}.{t}: {n:,}")
    except Exception as e:
        print(f"  {bronze_db}.{t}: erro -> {e}")
